In [ ]:
import pandas as pd
import json
import mne
import numpy as np
import mne
import os
import keras
from itertools import product
import glob

# --------------------------------------------------------------------------
# REPRODUCIBILITY & HARDWARE SETUP (Must be first)
# ---------------------------------------------------------------------------
print("it started")
import os
import random
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import numpy as np
import tensorflow as tf
import torch

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# ---------------------------------------------------------------------------
# ORIGINAL IMPORTS & SETUP
# ---------------------------------------------------------------------------
import json
import uuid
import pandas as pd
import matplotlib.pyplot as plt

# Scipy & MNE
import mne
from scipy.signal import stft, welch
from scipy.stats import entropy, norm
from sklearn.model_selection import KFold, train_test_split

# Scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras
from tensorflow.keras import layers, models, Model, callbacks

print(f"Reproducibility settings locked with SEED: {SEED}")

# GPU Check
if tf.config.list_physical_devices('GPU'):
    print("TensorFlow GPU Accelerated Backend Active.")
else:
    print("No GPU detected for TensorFlow. Using CPU.")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU Accelerated Backend Active: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")

it started
Reproducibility settings locked with SEED: 42
TensorFlow GPU Accelerated Backend Active.
CUDA GPU Accelerated Backend Active: Tesla T4


In [ ]:
# Path to the participants TSV file
file_path = "/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /participants.tsv"

# Read the tab-separated file
df = pd.read_csv(file_path, sep='\t')

# Display the first few rows
df.head()

,participant_id,subject_id,group,updrs_part_iii,updrs_total,moca,age,sex,disease_duration,ledd,pigd_score,td_score,ctt
0,sub-001,HC0001,HC,0.0,0.0,30.0,42.0,M,NaN,NaN,NaN,NaN,NaN
1,sub-002,HC0003,HC,2.0,3.0,27.0,60.0,M,NaN,NaN,NaN,NaN,66.0
2,sub-003,HC0004,HC,0.0,1.0,27.0,60.0,F,NaN,NaN,NaN,NaN,63.0
3,sub-004,HC0005,HC,1.0,1.0,25.0,72.0,M,NaN,NaN,NaN,NaN,116.0
4,sub-005,HC0006,HC,NaN,NaN,NaN,47.0,M,NaN,NaN,NaN,NaN,NaN


In [ ]:
sub_condition = df.iloc[:,2].values
print(sub_condition[0:5])

['HC' 'HC' 'HC' 'HC' 'HC']


In [ ]:
nan_counts = df.isna().sum()
print(nan_counts)

participant_id       0
subject_id           0
group                0
updrs_part_iii       5
updrs_total          5
moca                 4
age                  0
sex                  0
disease_duration    28
ledd                29
pigd_score          31
td_score            31
ctt                  9
dtype: int64


In [ ]:
print(df.shape)

(144, 13)


In [ ]:
missing_ids = []

for i in range(1, 145):
    sub_id = f"{i:03d}"
    file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-rest_eeg.set"

    # Check if file exists; if not, store or print i
    if not os.path.exists(file_path):
        print(i)
        missing_ids.append(i)

In [ ]:
missing_ids = []

for i in range(1, 145):
    # Format i with 3-digit zero-padding (e.g., 001, 002, ..., 144)
    sub_id = f"{i:03d}"
    file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-walk_eeg.set"

    # Check if the file does NOT exist and print i
    if not os.path.exists(file_path):
        print(i)
        missing_ids.append(i)

1
5
16
20
25
36
43
84
100
120
126


In [ ]:


# Format participant ID as 3-digit zero-padded string ('001')
sub_id = f"{1:03d}"
file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-rest_eeg.set"

# Load the file into memory
raw = mne.io.read_raw_eeglab(file_path, preload=True)

# Extract raw numerical array: shape is (Channels, Length)
signal = raw.get_data()

print("Signal shape (C, L):", signal.shape)

Signal shape (C, L): (65, 60964)


/tmp/ipykernel_7260/1231990765.py:6: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True)


In [ ]:
sfreq = raw.info['sfreq']

print(f"Sampling Frequency: {sfreq} Hz")

Sampling Frequency: 250.0 Hz


In [ ]:

def get_eeg_signal(sub_id, task="walk", band="full",target_sfreq=256,duration=2.0, notch_freq=50.0,
    reject_threshold=0.00028,
    base_dir="/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 "
):
    """
    Loads, cleans, filters by frequency band, resamples, and segments EEG data.
    Returns float32 NumPy array of shape (N_epochs, Channels, Time).
    """
    # 1. Map band names to frequency limits
    band_limits = {
        'full':  (1.0, 45.0),
        'delta': (1.0, 4.0),
        'theta': (4.0, 8.0),
        'alpha': (8.0, 12.0),
        'beta':  (12.0, 30.0),
        'gamma': (30.0, 45.0)
    }

    if band.lower() not in band_limits:
        raise ValueError(f"Invalid band '{band}'. Choose from: {list(band_limits.keys())}")

    l_freq, h_freq = band_limits[band.lower()]

    # 2. Format subject ID
    if isinstance(sub_id, int):
        sub_str = f"{sub_id:03d}"
    else:
        sub_str = str(sub_id).zfill(3)

    file_path = f"{base_dir}/sub-{sub_str}/eeg/sub-{sub_str}_task-{task}_eeg.set"

    # 3. Load continuous file
    raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)

    # 4. Fix channel types & isolate EEG
    channel_type_mapping = {
        'EOG1': 'eog', 'EOG2': 'eog', 'EOG3': 'eog', 'EOG4': 'eog', 'VREF': 'misc'
    }
    existing_mapping = {ch: t for ch, t in channel_type_mapping.items() if ch in raw.ch_names}
    if existing_mapping:
        raw.set_channel_types(existing_mapping)

    raw.pick_types(eeg=True, eog=False, misc=False)

    # 5. PREPROCESSING
    # A. Bandpass filter for selected band
    raw.filter(l_freq=l_freq, h_freq=h_freq, fir_design='firwin', verbose=False)

    # B. Notch Filter (only if applicable to selected band range)
    if notch_freq is not None and h_freq >= notch_freq:
        raw.notch_filter(freqs=notch_freq, verbose=False)

    # C. Common Average Reference (CAR)
    raw.set_eeg_reference(ref_channels='average', projection=False, verbose=False)

    # 6. Resample to target frequency (e.g., 128 Hz)
    raw.resample(sfreq=target_sfreq, verbose=False)

    # 7. Epoching with Artifact Rejection
    events = mne.make_fixed_length_events(raw, duration=duration)
    reject_criteria = dict(eeg=reject_threshold) if reject_threshold is not None else None

    epochs = mne.Epochs(
        raw,
        events=events,
        tmin=0,
        tmax=duration - (1 / target_sfreq),  # Exactly 256 samples at 128 Hz
        baseline=None,
        reject=reject_criteria,
        preload=True,
        verbose=False
    )

    # Extract array and cast to float32 to save RAM
    signal = epochs.get_data().astype(np.float32)

    return signal


# --- Example Usage ---
# Extract Beta band for walking task at 128 Hz
beta_walk = get_eeg_signal(sub_id=3, task="walk", band="beta")

# Extract Full spectrum (1-45 Hz) for resting task at 128 Hz
full_rest = get_eeg_signal(sub_id=3, task="rest", band="full")

print("Beta Walk Signal shape (N, C, T):", beta_walk.shape)  # e.g., (N, 60, 256)
print("Full Rest Signal shape (N, C, T):", full_rest.shape)  # e.g., (N, 60, 256)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_7260/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_7260/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_7260/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_7260/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


Beta Walk Signal shape (N, C, T): (121, 60, 512)
Full Rest Signal shape (N, C, T): (121, 60, 512)


In [ ]:
non_walk_ids = [1,5,16,20,25,36,43,84,100,120,126]

In [ ]:
rest_ids = [i for i in range(1,145)]
walk_ids = [i for i in range(1,145) if i not in non_walk_ids]

In [ ]:
def get_data(task, band, rest_ids, walk_ids):
    X_hc = []
    X_pd = []

    # Select subject list based on task
    sub_ids = rest_ids if task == "rest" else walk_ids

    for sub_id in sub_ids:
        try:
            # Extract signal for the current subject
            eeg_signal = get_eeg_signal(sub_id=sub_id, task=task, band=band)

            # Split into HC (< 29) or PD (>= 29)
            if sub_id < 29:
                X_hc.append(eeg_signal)
            else:
                X_pd.append(eeg_signal)

        except Exception as e:
            print(f"Skipping Subject {sub_id} ({task}, {band}) due to error: {e}")

    return X_hc, X_pd


In [ ]:
def balance_matrices_subject_wise(X_list_c0, X_list_c1):
    c0_windows_per_sub = [sub.shape[0] for sub in X_list_c0]
    c1_windows_per_sub = [sub.shape[0] for sub in X_list_c1]

    total_c0 = sum(c0_windows_per_sub)
    total_c1 = sum(c1_windows_per_sub)

    if total_c0 == total_c1:
        return np.concatenate(X_list_c0, axis=0), np.concatenate(X_list_c1, axis=0)

    if total_c1 > total_c0:
        maj_list = X_list_c1
        maj_counts = np.array(c1_windows_per_sub)
        target_total = total_c0
        is_c1_majority = True
    else:
        maj_list = X_list_c0
        maj_counts = np.array(c0_windows_per_sub)
        target_total = total_c1
        is_c1_majority = False

    num_maj_subs = len(maj_list)
    allocations = np.zeros(num_maj_subs, dtype=int)
    remaining_target = target_total
    active_subs = np.ones(num_maj_subs, dtype=bool)

    while remaining_target > 0 and np.any(active_subs):
        num_active = np.sum(active_subs)
        base_share = remaining_target // num_active
        remainder = remaining_target % num_active

        if base_share == 0:
            chosen_indices = np.where(active_subs)[0][:remaining_target]
            for idx in chosen_indices:
                allocations[idx] += 1
            break

        for i in range(num_maj_subs):
            if active_subs[i]:
                share = base_share + (1 if remainder > 0 else 0)
                remainder -= 1 if remainder > 0 else 0

                available = maj_counts[i] - allocations[i]
                take = min(share, available)

                allocations[i] += take
                remaining_target -= take

                if allocations[i] == maj_counts[i]:
                    active_subs[i] = False

    processed_maj_list = []
    rng = np.random.default_rng(SEED)
    for i, sub_windows in enumerate(maj_list):
        n_needed = allocations[i]
        if n_needed > 0:
            chosen_indices = rng.choice(sub_windows.shape[0], size=n_needed, replace=False)
            processed_maj_list.append(sub_windows[chosen_indices])

    X_processed_maj = np.concatenate(processed_maj_list, axis=0)

    if is_c1_majority:
        return np.concatenate(X_list_c0, axis=0), X_processed_maj
    else:
        return X_processed_maj, np.concatenate(X_list_c1, axis=0)

In [ ]:
def scale_data(X_list):
    scaled = []
    for sub in X_list:
        flat = sub.reshape(-1, sub.shape[-1])
        mu = np.mean(flat, axis=0)
        std = np.std(flat, axis=0) + 1e-8
        scaled.append((sub - mu) / std)
    return scaled

In [ ]:
class ChannelAttention(layers.Layer):
    def __init__(self, channels):
        super(ChannelAttention, self).__init__()
        self.attn = layers.Dense(channels, activation='softmax')

    def call(self, x):
        avg_pool = tf.reduce_mean(x, axis=1)
        weights = self.attn(avg_pool)
        weights = tf.expand_dims(weights, 1)
        return x * weights

In [ ]:
def Conv_block(input_tensor, F1=16, D=2, kernel_size=64, dropout=0.3):
    """
    Corrected ATCNet Conv Block with proper (Channels, Time, Filters) dimensions.
    """
    # 1. First Temporal Conv along the time axis (kernel height=1, width=kernel_size)
    x = layers.Conv2D(F1, (1, kernel_size), padding='same', use_bias=False)(input_tensor)
    x = layers.BatchNormalization()(x)

    # 2. Spatial Depthwise Conv across EEG Channels (kernel height=n_chans, width=1)
    # input_tensor shape: (Batch, Channels, Time, 1)
    n_chans = input_tensor.shape[1]
    x = layers.DepthwiseConv2D(
        (n_chans, 1),
        depth_multiplier=D,
        use_bias=False,
        depthwise_constraint=keras.constraints.max_norm(1.)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)

    # 3. Pooling along the temporal axis
    x = layers.AveragePooling2D((1, 8))(x)
    x = layers.Dropout(dropout)(x)

    return x


def TCN_block(input_tensor, input_dim=32, kernel_size=4, dropout=0.3, dilation_rate=1):
    """
    Official Causal Temporal Convolutional Network Block with Residual Connection.
    """
    x = layers.Conv1D(input_dim, kernel_size, padding='causal',
                      dilation_rate=dilation_rate, activation='elu')(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Conv1D(input_dim, kernel_size, padding='causal',
                      dilation_rate=dilation_rate, activation='elu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)

    if input_tensor.shape[-1] != input_dim:
        res = layers.Conv1D(input_dim, 1, padding='same')(input_tensor)
    else:
        res = input_tensor

    return layers.add([x, res])


def create_atcnet_official(
    input_shape=(22, 1000),
    nb_classes=1,
    F1=16,
    D=2,
    kernel_size=64,
    eeg_dropout=0.3,
    tcn_dropout=0.3,
    tcn_kernel_size=4,
    n_windows=5,
    key_dim=8,
    num_heads=2
):
    """
    Official ATCNet Implementation with corrected dimension permutations.
    Expects input_shape as (Channels, Timepoints).
    """
    inputs = layers.Input(shape=input_shape)

    # 1. Reshape & Permute to standard 4D tensor: (Batch, Channels, Timepoints, 1)
    if len(input_shape) == 2:
        x = layers.Reshape((input_shape[0], input_shape[1], 1))(inputs)
    else:
        x = inputs

    # 2. Convolutional Feature Extractor
    x = Conv_block(x, F1=F1, D=D, kernel_size=kernel_size, dropout=eeg_dropout)

    # 3. Squeeze spatial height (Channels dimension becomes 1 after DepthwiseConv)
    # Output shape becomes: (Batch, Reduced_Timepoints, Filters)
    x = layers.Reshape((-1, F1 * D))(x)

    # 4. Multi-Window Segmentation & Processing
    time_steps = x.shape[1]
    window_size = time_steps // n_windows if time_steps is not None else 10

    dense_outputs = []

    for i in range(n_windows):
        start_idx = i * (window_size // 2)
        end_idx = start_idx + window_size

        # Slicing temporal window
        x_win = x[:, start_idx:end_idx, :] if time_steps is not None else x

        # Multi-Head Attention
        norm_win = layers.LayerNormalization(epsilon=1e-6)(x_win)
        attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(norm_win, norm_win)
        attn_out = layers.Dropout(0.1)(attn_out)
        x_att = layers.add([x_win, attn_out])

        # Temporal Convolutional Network Block
        x_tcn = TCN_block(x_att, input_dim=F1 * D, kernel_size=tcn_kernel_size,
                          dropout=tcn_dropout, dilation_rate=1)
        x_tcn = TCN_block(x_tcn, input_dim=F1 * D, kernel_size=tcn_kernel_size,
                          dropout=tcn_dropout, dilation_rate=2)

        win_feat = layers.Flatten()(x_tcn)
        dense_outputs.append(win_feat)

    # 5. Concatenate multi-window representations
    if len(dense_outputs) > 1:
        x_concat = layers.Concatenate(axis=-1)(dense_outputs)
    else:
        x_concat = dense_outputs[0]

    # 6. Classification Head
    activation = 'sigmoid' if nb_classes == 1 else 'softmax'
    outputs = layers.Dense(nb_classes, activation=activation)(x_concat)

    return Model(inputs=inputs, outputs=outputs, name="ATCNet_Official")

In [ ]:
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from itertools import product

def format_eeg_tensor_atcnet(data_array):
    """
    Formats input EEG data array into standard 3D matrix for ATCNet:
    (Batch/Epochs, Channels, Timepoints).
    """
    arr = np.asarray(data_array, dtype=np.float32)

    if arr.ndim == 2:
        # Single Epoch (Channels, Time) -> (1, Channels, Time)
        return np.expand_dims(arr, axis=0)
    elif arr.ndim == 3:
        # Check if shape is (Epochs, Time, Channels) and transpose if necessary
        # ATCNet expects (Epochs, Channels, Time)
        if arr.shape[1] > arr.shape[2]:  # If Time > Channels in axis 1
            return np.transpose(arr, (0, 2, 1))
        return arr
    elif arr.ndim == 4:
        # Remove singleton dimensions e.g. (Epochs, 1, Channels, Time)
        arr = np.squeeze(arr)
        if arr.ndim == 2:
            return np.expand_dims(arr, axis=0)
        elif arr.ndim == 3 and arr.shape[1] > arr.shape[2]:
            return np.transpose(arr, (0, 2, 1))
        return arr
    else:
        raise ValueError(f"Unexpected array dimension: {arr.ndim} (shape: {arr.shape})")

def run_subject_level_mc_cv_atcnet(X_healthy, X_pd, SEED=42, epochs=50):
    X_healthy = scale_data(X_healthy)
    X_pd = scale_data(X_pd)

    # Infer input shape from single epoch: (Channels, Timepoints)
    sample_sub = X_healthy[0]
    sample_epoch = sample_sub[0] if sample_sub.ndim == 3 else sample_sub

    if sample_epoch.ndim == 2:
        # Ensure (Channels, Timepoints) order
        if sample_epoch.shape[0] > sample_epoch.shape[1]:
            input_shape = (sample_epoch.shape[1], sample_epoch.shape[0])
        else:
            input_shape = (sample_epoch.shape[0], sample_epoch.shape[1])
    elif sample_epoch.ndim == 3:
        sample_epoch = np.squeeze(sample_epoch)
        if sample_epoch.shape[0] > sample_epoch.shape[1]:
            input_shape = (sample_epoch.shape[1], sample_epoch.shape[0])
        else:
            input_shape = (sample_epoch.shape[0], sample_epoch.shape[1])
    else:
        raise ValueError(f"Unexpected epoch shape: {sample_epoch.shape}")

    print(f"--> Inferred ATCNet Input Shape (Channels, Timepoints): {input_shape}")

    n_hc, n_pd = len(X_healthy), len(X_pd)
    outer_kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    thresholds = list(range(65, 95, 5))

    hc_splits = list(outer_kf.split(np.arange(n_hc)))
    pd_splits = list(outer_kf.split(np.arange(n_pd)))

    total_correct = 0
    total_subjects = 0
    fold_summary_records = []

    # ATCNet Hyperparameter Grid Search
    param_grid = {
        'lr': [1e-3],
        'batch_size': [32],
        'F1': [16],
        'D': [2],
        'n_windows': [5],
        'num_heads': [2]
    }

    keys = param_grid.keys()
    all_combinations = [dict(zip(keys, combo)) for combo in product(*param_grid.values())]

    for fold in range(5):
        print(f"\n========================================")
        print(f"========== OUTER FOLD {fold+1} / 5 ==========")
        print(f"========================================")

        hc_train_all, hc_test = hc_splits[fold]
        pd_train_all, pd_test = pd_splits[fold]

        best_score = -1.0
        best_params = None
        best_threshold = 75

        hc_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(hc_train_all))
        pd_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(pd_train_all))

        for params in all_combinations:
            inner_fold_accuracies = []
            inner_fold_thresholds = []

            for inner_fold in range(3):
                tf.keras.backend.clear_session()

                hc_tr_in_idx, hc_val_in_idx = hc_inner_splits[inner_fold]
                pd_tr_in_idx, pd_val_in_idx = pd_inner_splits[inner_fold]

                hc_train_sub = [X_healthy[hc_train_all[i]] for i in hc_tr_in_idx]
                pd_train_sub = [X_pd[pd_train_all[i]] for i in pd_tr_in_idx]
                hc_val_sub = [X_healthy[hc_train_all[i]] for i in hc_val_in_idx]
                pd_val_sub = [X_pd[pd_train_all[i]] for i in pd_val_in_idx]

                # 1. Balance classes for inner training (Subject-Level)
                X_tr_hc_bal, X_tr_pd_bal = balance_matrices_subject_wise(hc_train_sub, pd_train_sub)
                X_tr = np.concatenate([X_tr_hc_bal, X_tr_pd_bal], axis=0)
                y_tr = np.concatenate([np.zeros(len(X_tr_hc_bal)), np.ones(len(X_tr_pd_bal))], axis=0)

                # Format training data to 3D tensor
                X_tr = format_eeg_tensor_atcnet(X_tr)

                # Shuffle training epochs safely with dynamic seed
                rng = np.random.RandomState(SEED + inner_fold)
                shuffle_idx = rng.permutation(len(X_tr))
                X_tr = X_tr[shuffle_idx]
                y_tr = y_tr[shuffle_idx]

                # Instantiate Official ATCNet Model
                inner_model = create_atcnet_official(
                    input_shape=input_shape,
                    nb_classes=1,
                    F1=params['F1'],
                    D=params['D'],
                    n_windows=params['n_windows'],
                    num_heads=params['num_heads']
                )
                inner_model.compile(
                    optimizer=tf.keras.optimizers.Adam(learning_rate=params['lr']),
                    loss='binary_crossentropy',
                    metrics=['accuracy']
                )

                # Fit Inner Model (Fixed Epochs, No Early Stopping / Validation Split)
                inner_model.fit(
                    X_tr, y_tr,
                    epochs=epochs,
                    batch_size=params['batch_size'],
                    verbose=0
                )

                # Inner validation threshold tuning
                val_subjects = hc_val_sub + pd_val_sub
                val_labels = [0] * len(hc_val_sub) + [1] * len(pd_val_sub)

                val_subject_ratios = []
                valid_val_labels = []

                for sub, true_lbl in zip(val_subjects, val_labels):
                    sub_array = format_eeg_tensor_atcnet(sub)

                    if sub_array.shape[0] == 0:
                        continue

                    epoch_probs = inner_model.predict(sub_array, batch_size=params['batch_size'], verbose=0).flatten()
                    pct_pd = float(np.mean(epoch_probs) * 100)
                    val_subject_ratios.append(pct_pd)
                    valid_val_labels.append(true_lbl)

                best_t_inner, max_inner_acc = 75, -1.0
                for t in thresholds:
                    t_preds = [1 if ratio >= t else 0 for ratio in val_subject_ratios]
                    acc = accuracy_score(valid_val_labels, t_preds) if len(valid_val_labels) > 0 else 0.0
                    if acc > max_inner_acc:
                        max_inner_acc = acc
                        best_t_inner = t

                inner_fold_accuracies.append(max_inner_acc)
                inner_fold_thresholds.append(best_t_inner)

            mean_inner_acc = np.mean(inner_fold_accuracies)
            if mean_inner_acc > best_score:
                best_score = mean_inner_acc
                best_params = params
                best_threshold = int(np.median(inner_fold_thresholds))

        print(f">> Best Grid Parameters Selected: {best_params} | Threshold: {best_threshold}% (Inner Acc: {best_score:.4f})")

        # --- OUTER TRAINING & FINAL FIT ---
        tf.keras.backend.clear_session()

        hc_train_final = [X_healthy[i] for i in hc_train_all]
        pd_train_final = [X_pd[i] for i in pd_train_all]

        X_tr_hc_bal, X_tr_pd_bal = balance_matrices_subject_wise(hc_train_final, pd_train_final)
        X_tr_f = np.concatenate([X_tr_hc_bal, X_tr_pd_bal], axis=0)
        y_tr_f = np.concatenate([np.zeros(len(X_tr_hc_bal)), np.ones(len(X_tr_pd_bal))], axis=0)

        X_tr_f = format_eeg_tensor_atcnet(X_tr_f)
        shuffle_idx_final = np.random.RandomState(SEED).permutation(len(X_tr_f))
        X_tr_f, y_tr_f = X_tr_f[shuffle_idx_final], y_tr_f[shuffle_idx_final]

        final_model = create_atcnet_official(
            input_shape=input_shape,
            nb_classes=1,
            F1=best_params['F1'],
            D=best_params['D'],
            n_windows=best_params['n_windows'],
            num_heads=best_params['num_heads']
        )
        final_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['lr']),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        # Track Training Time (Fixed Epochs, No Early Stopping)
        start_time = time.time()
        final_model.fit(
            X_tr_f, y_tr_f,
            epochs=epochs,
            batch_size=best_params['batch_size'],
            verbose=0
        )
        training_time = time.time() - start_time

        # Track Model Parameters Count (Placed safely AFTER fit)
        model_params_count = final_model.count_params()

        test_subjects = [X_healthy[i] for i in hc_test] + [X_pd[i] for i in pd_test]
        test_labels = [0] * len(hc_test) + [1] * len(pd_test)
        n_hc_test = len(hc_test)
        n_pd_test = len(pd_test)

        hc_correct_count = 0
        pd_correct_count = 0

        min_t, max_t = thresholds[0], thresholds[-1]
        for sub, true_label in zip(test_subjects, test_labels):
            sub_array = format_eeg_tensor_atcnet(sub)

            if sub_array.shape[0] == 0:
                continue

            pct_pd = float(np.mean(final_model.predict(sub_array, batch_size=best_params['batch_size'], verbose=0).flatten()) * 100)

            vote_thresholds = [
                max(min_t, best_threshold - 5),
                best_threshold,
                min(max_t, best_threshold + 5)
            ]
            votes = [1 if pct_pd >= t else 0 for t in vote_thresholds]
            pred = 1 if sum(votes) >= 2 else 0

            if pred == true_label:
                if true_label == 0:
                    hc_correct_count += 1
                else:
                    pd_correct_count += 1

        fold_total_correct = hc_correct_count + pd_correct_count
        fold_total_subjects = len(test_subjects)

        total_correct += fold_total_correct
        total_subjects += fold_total_subjects

        fold_acc = (fold_total_correct / fold_total_subjects) * 100 if fold_total_subjects > 0 else 0.0

        fold_summary_records.append({
            'Fold Number': fold + 1,
            'Optimal Hyperparams': str(best_params),
            'Optimal Threshold (%)': best_threshold,
            'Healthy Correct': f"{hc_correct_count}/{n_hc_test}",
            'PD Correct': f"{pd_correct_count}/{n_pd_test}",
            'Fold Accuracy (%)': f"{fold_acc:.2f}%",
            'Total Correct': f"{fold_total_correct}/{fold_total_subjects}",
            'Training Time (s)': f"{training_time:.2f}",
            'Model Parameters': f"{model_params_count:,}"
        })

        print(f"Outer Fold {fold+1} Stats -> Healthy: {hc_correct_count}/{n_hc_test} | PD: {pd_correct_count}/{n_pd_test} | Acc: {fold_acc:.2f}% | Time: {training_time:.2f}s | Params: {model_params_count:,}")

    summary_df = pd.DataFrame(fold_summary_records)
    overall_acc = (total_correct / total_subjects) * 100 if total_subjects > 0 else 0.0

    print(f"\n========================================")
    print(f"Total Combined Correct: {total_correct}/{total_subjects}")
    print(f"Overall Nested Cross-Validation Accuracy: {overall_acc:.2f}%")
    print("\n--- Nested Cross-Validation Summary ---")
    print(summary_df.to_string(index=False))

    return summary_df

In [ ]:
task = 'rest'
band = 'alpha'
X_hc,X_pd = get_data(task, band, rest_ids, walk_ids)
df = run_subject_level_mc_cv_atcnet(X_hc, X_pd, SEED=42)
print(df)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_7260/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_7260/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_7260/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_7260/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


...

Outer Fold 4 Stats -> Healthy: 4/5 | PD: 20/23 | Acc: 85.71% | Time: 476.44s | Params: 101,137

========== OUTER FOLD 5 / 5 ==========


2026-09-09 13:53:03.830490: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
E0000 00:00:1788962004.768674    7260 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/ATCNet_Official_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2026-09-09 13

>> Best Grid Parameters Selected: {'lr': 0.001, 'batch_size': 32, 'F1': 16, 'D': 2, 'n_windows': 5, 'num_heads': 2} | Threshold: 80% (Inner Acc: 0.7933)


2026-09-09 14:10:01.689661: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
E0000 00:00:1788963022.792063    7260 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/ATCNet_Official_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2026-09-09 14

Outer Fold 5 Stats -> Healthy: 3/5 | PD: 15/23 | Acc: 64.29% | Time: 481.26s | Params: 101,137

Total Combined Correct: 112/144
Overall Nested Cross-Validation Accuracy: 77.78%

--- Nested Cross-Validation Summary ---
 Fold Number                                                               Optimal Hyperparams  Optimal Threshold (%) Healthy Correct PD Correct Fold Accuracy (%) Total Correct Training Time (s) Model Parameters
           1 {'lr': 0.001, 'batch_size': 32, 'F1': 16, 'D': 2, 'n_windows': 5, 'num_heads': 2}                     65             5/6      21/24            86.67%         26/30            448.71          101,137
           2 {'lr': 0.001, 'batch_size': 32, 'F1': 16, 'D': 2, 'n_windows': 5, 'num_heads': 2}                     70             4/6      20/23            82.76%         24/29            449.22          101,137
           3 {'lr': 0.001, 'batch_size': 32, 'F1': 16, 'D': 2, 'n_windows': 5, 'num_heads': 2}                     65             6/6      14/23  

In [ ]:
print(df)

   Fold Number                                Optimal Hyperparams  \
0            1  {'lr': 0.001, 'batch_size': 32, 'F1': 16, 'D':...   
1            2  {'lr': 0.001, 'batch_size': 32, 'F1': 16, 'D':...   
2            3  {'lr': 0.001, 'batch_size': 32, 'F1': 16, 'D':...   
3            4  {'lr': 0.001, 'batch_size': 32, 'F1': 16, 'D':...   
4            5  {'lr': 0.001, 'batch_size': 32, 'F1': 16, 'D':...   

   Optimal Threshold (%) Healthy Correct PD Correct Fold Accuracy (%)  \
0                     65             5/6      21/24            86.67%   
1                     70             4/6      20/23            82.76%   
2                     65             6/6      14/23            68.97%   
3                     65             4/5      20/23            85.71%   
4                     80             3/5      15/23            64.29%   

  Total Correct Training Time (s) Model Parameters  
0         26/30            448.71          101,137  
1         24/29            449.22       

In [ ]:
task = 'walk'
band = 'alpha'
X_hc,X_pd = get_data(task, band, rest_ids, walk_ids)
df = run_subject_level_mc_cv_atcnet(X_hc, X_pd, SEED=42)
print(df)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_7260/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_7260/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_7260/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_7260/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


...

Outer Fold 4 Stats -> Healthy: 3/4 | PD: 17/22 | Acc: 76.92% | Time: 367.82s | Params: 101,137

========== OUTER FOLD 5 / 5 ==========


2026-09-09 15:36:26.210032: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
E0000 00:00:1788968207.591979    7260 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/ATCNet_Official_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2026-09-09 15

>> Best Grid Parameters Selected: {'lr': 0.001, 'batch_size': 32, 'F1': 16, 'D': 2, 'n_windows': 5, 'num_heads': 2} | Threshold: 65% (Inner Acc: 0.7871)


2026-09-09 15:50:09.902099: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
E0000 00:00:1788969031.195397    7260 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/ATCNet_Official_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2026-09-09 15

Outer Fold 5 Stats -> Healthy: 2/4 | PD: 16/22 | Acc: 69.23% | Time: 385.82s | Params: 101,137

Total Combined Correct: 95/133
Overall Nested Cross-Validation Accuracy: 71.43%

--- Nested Cross-Validation Summary ---
 Fold Number                                                               Optimal Hyperparams  Optimal Threshold (%) Healthy Correct PD Correct Fold Accuracy (%) Total Correct Training Time (s) Model Parameters
           1 {'lr': 0.001, 'batch_size': 32, 'F1': 16, 'D': 2, 'n_windows': 5, 'num_heads': 2}                     65             3/5      19/22            81.48%         22/27            368.17          101,137
           2 {'lr': 0.001, 'batch_size': 32, 'F1': 16, 'D': 2, 'n_windows': 5, 'num_heads': 2}                     65             2/5      18/22            74.07%         20/27            343.66          101,137
           3 {'lr': 0.001, 'batch_size': 32, 'F1': 16, 'D': 2, 'n_windows': 5, 'num_heads': 2}                     65             1/5      14/22   